# Cross Dataset Experiment 

In this notebook, we compare the performance of three models: Random Forest Classifier, Logistic Regression and XGBoost against our 3 cleaned datasets. We also applied class weighting across all models to address class imbalance. The results of each experiment is recorded and will evaluated at the end of this notebook.

In [1]:
# Load cleaned datasets
import pandas as pd

df1 = pd.read_csv("data/cleaned/dataset_1/dataset_1_clean.csv")
df2 = pd.read_csv("data/cleaned/dataset_2/dataset_2_clean.csv")
df3 = pd.read_csv("data/cleaned/dataset_3/dataset_3_clean.csv")

## Feature Alignment

In [2]:
# Find common columns across all three datasets
common_cols = [col for col in df1.columns if col in df2.columns and col in df3.columns]

print("Number of common columns:", len(common_cols))

# Keep only common columns in each dataset
df1 = df1[common_cols].copy()
df2 = df2[common_cols].copy()
df3 = df3[common_cols].copy()

print(df1.columns.tolist() == df2.columns.tolist())
print(df1.columns.tolist() == df3.columns.tolist())

Number of common columns: 14
True
True


## Data Splitting

In [3]:
from sklearn.model_selection import train_test_split

def split_data(df):
    X = df.drop("target", axis=1)
    y = df["target"]
    
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X1_train, X1_test, y1_train, y1_test = split_data(df1)
X2_train, X2_test, y2_train, y2_test = split_data(df2)
X3_train, X3_test, y3_train, y3_test = split_data(df3)

In [4]:
numeric_features = X1_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X1_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['bp', 'sg', 'al', 'su', 'bu', 'sc', 'sod', 'pot', 'hemo']
Categorical features: ['rbc', 'wc', 'rc', 'htn']


/var/folders/s1/z7ywf1314wggd5t1h2rl_t7w0000gn/T/ipykernel_12426/1778746348.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X1_train.select_dtypes(include=["object"]).columns.tolist()


## Preprocessing Pipeline

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [6]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Function to compute all metrics at once
def get_metrics(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    return {
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob)
    }

In [7]:
# Initialize results list
results = []

---
# Dataset 1 Experiments
---

## Preprocessing for Dataset 1

In [8]:
# Fit preprocessor on D1 ONLY
X1_train_t = preprocessor.fit_transform(X1_train)

# Transform ALL test sets using SAME preprocessor
X1_test_t = preprocessor.transform(X1_test)
X2_test_t = preprocessor.transform(X2_test)
X3_test_t = preprocessor.transform(X3_test)

# Experiment 1
- Training Dataset: 1
- Model: Random Forest Classifer
- Imbalance Handling: None


In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X1_train_t, y1_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 1, 'test_dataset': test_set, 'model': 'Random Forest', 'imbalance_handling': 'None', **get_metrics(model, X_test, y_test)})

# Experiment 2
- Training Dataset: 1
- Model: Random Forest Classifier 
- Imbalance Handling: Weights

In [10]:
model = RandomForestClassifier(class_weight="balanced", random_state=42)
model.fit(X1_train_t, y1_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 1, 'test_dataset': test_set, 'model': 'Random Forest', 'imbalance_handling': 'Balanced', **get_metrics(model, X_test, y_test)})

## Experiment 3
- Training Dataset: 1
- Model: Logistic Regression
- Imbalance Handling: None

In [11]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X1_train_t, y1_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 1, 'test_dataset': test_set, 'model': 'Logistic Regression', 'imbalance_handling': 'None', **get_metrics(model, X_test, y_test)})

## Experiment 4
- Training Dataset: 1
- Model: Logistic Regression
- Imbalance Handling: Weights

In [12]:
model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X1_train_t, y1_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 1, 'test_dataset': test_set, 'model': 'Logistic Regression', 'imbalance_handling': 'Balanced', **get_metrics(model, X_test, y_test)})

## Experiment 5
- Training Dataset: 1
- Model: XGBoost
- Imbalance Handling: None

In [13]:
from xgboost import XGBClassifier

model = XGBClassifier(random_state=42, eval_metric="logloss")
model.fit(X1_train_t, y1_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 1, 'test_dataset': test_set, 'model': 'XGBoost', 'imbalance_handling': 'None', **get_metrics(model, X_test, y_test)})

## Experiment 6
- Training Dataset: 1
- Model: XGBoost
- Imbalance Handling: Weights

In [14]:
neg = (y1_train == 0).sum()
pos = (y1_train == 1).sum()

model = XGBClassifier(
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=42
)
model.fit(X1_train_t, y1_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 1, 'test_dataset': test_set, 'model': 'XGBoost', 'imbalance_handling': 'Balanced', **get_metrics(model, X_test, y_test)})

---
## Dataset 2 Experiments
---

## Preprocessing for Dataset 2

In [15]:
# Fit preprocessor on D2 ONLY
X2_train_t = preprocessor.fit_transform(X2_train)

# Transform ALL test sets using SAME preprocessor
X1_test_t = preprocessor.transform(X1_test)
X2_test_t = preprocessor.transform(X2_test)
X3_test_t = preprocessor.transform(X3_test)

In [16]:
# Fit preprocessor on D2 ONLY
X2_train_t = preprocessor.fit_transform(X2_train)

# Transform ALL test sets using SAME preprocessor
X1_test_t = preprocessor.transform(X1_test)
X2_test_t = preprocessor.transform(X2_test)
X3_test_t = preprocessor.transform(X3_test)

## Experiment 1
- Training Dataset: 1
- Model: Random Forest Classifier
- Imbalance Handling: None

In [17]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X2_train_t, y2_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 2, 'test_dataset': test_set, 'model': 'Random Forest', 'imbalance_handling': 'None', **get_metrics(model, X_test, y_test)})

## Experiment 2
- Training Dataset: 1
- Model: Random Forest Classifier
- Imbalance Handling: Weights

In [18]:
model = RandomForestClassifier(class_weight="balanced", random_state=42)
model.fit(X2_train_t, y2_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 2, 'test_dataset': test_set, 'model': 'Random Forest', 'imbalance_handling': 'Balanced', **get_metrics(model, X_test, y_test)})

## Experiment 3
- Training Dataset: 1
- Model: Logistic Regression
- Imbalance Handling: None

In [19]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X2_train_t, y2_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 2, 'test_dataset': test_set, 'model': 'Logistic Regression', 'imbalance_handling': 'None', **get_metrics(model, X_test, y_test)})

## Experiment 4
- Training Dataset: 1
- Model: Logistic Regression
- Imbalance Handling: Weights

In [20]:
model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X2_train_t, y2_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 2, 'test_dataset': test_set, 'model': 'Logistic Regression', 'imbalance_handling': 'Balanced', **get_metrics(model, X_test, y_test)})

## Experiment 5
- Training Dataset: 1
- Model: XGBoost
- Imbalance Handling: None

In [21]:
from xgboost import XGBClassifier

model = XGBClassifier(random_state=42, eval_metric="logloss")
model.fit(X2_train_t, y2_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 2, 'test_dataset': test_set, 'model': 'XGBoost', 'imbalance_handling': 'None', **get_metrics(model, X_test, y_test)})

## Experiment 6
- Training Dataset: 1
- Model: XGBoost
- Imbalance Handling: Weights

In [22]:
neg = (y2_train == 0).sum()
pos = (y2_train == 1).sum()

model = XGBClassifier(
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=42
)
model.fit(X2_train_t, y2_train)

for test_set, X_test, y_test in [(1, X1_test_t, y1_test), (2, X2_test_t, y2_test), (3, X3_test_t, y3_test)]:
    results.append({'train_dataset': 2, 'test_dataset': test_set, 'model': 'XGBoost', 'imbalance_handling': 'Balanced', **get_metrics(model, X_test, y_test)})

---
## Dataset 3 Experiments
___

## Preprocessing for Dataset 3

## Experiment 1
- Training Dataset: 3
- Model: Random Forest Classifier
- Imbalance Handling: None

## Experiment 2
- Training Dataset: 3
- Model: Random Forest Classifier
- Imbalance Handling: Balanced

## Experiment 3
- Training Dataset: 3
- Model: Logistic Regression
- Imbalance Handling: None

## Experiment 4
- Training Dataset: 3
- Model: Logistic Regression
- Imbalance Handling: Balanced

## Experiment 5
- Training Dataset: 3
- Model: XGBoost
- Imbalance Handling: None

## Experiment 6
- Training Dataset: 3
- Model: XGBoost
- Imbalance Handling: Balanced

---
## Results Summary

In [23]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df

,train_dataset,test_dataset,model,imbalance_handling,accuracy,precision,recall,f1,roc_auc
0,1,1,Random Forest,None,1.000000,1.000000,1.000000,1.000000,1.000000
1,1,2,Random Forest,None,0.950000,0.925926,1.000000,0.961538,1.000000
2,1,3,Random Forest,None,0.199854,0.199854,1.000000,0.333130,0.522739
3,1,1,Random Forest,Balanced,1.000000,1.000000,1.000000,1.000000,1.000000
4,1,2,Random Forest,Balanced,0.962500,0.943396,1.000000,0.970874,1.000000
5,1,3,Random Forest,Balanced,0.199854,0.199854,1.000000,0.333130,0.527355
6,1,1,Logistic Regression,None,1.000000,1.000000,1.000000,1.000000,1.000000
7,1,2,Logistic Regression,None,1.000000,1.000000,1.000000,1.000000,1.000000
8,1,3,Logistic Regression,None,0.202532,0.199804,0.995128,0.332790,0.524773
9,1,1,Logistic Regression,Balanced,1.000000,1.000000,1.000000,1.000000,1.000000
